# FlightWise: Evidence-Grounded Airline Disruption and Passenger Policy Assistant

**Generative AI - Assignment 3: Group 4 Mini-Project**

**Group members**
- Aishwarya Nevrekar - 27PGAI0028
- Deepanshi Bansal - 27PGAI0110
- Kushagra Gupta - 27PGAI0115
- Kaushik Gadipelly - 

FlightWise is a Retrieval-Augmented Generation (RAG) system built on a custom dataset of official passenger-policy sources from 12 airlines plus DGCA/MoCA. The notebook intentionally reuses the syntax and workflow from the six in-class labs as much as possible.

## What from the in-class labs is reused?

| In-class lab | FlightWise use |
|---|---|
| Lab 1 - LLM Generation Parameters | Factual generation with low randomness |
| Lab 2 - Introduction to LangChain | `init_chat_model`, prompt templates and runnable chains |
| Lab 3 - Structured Output | Pydantic schema + `with_structured_output` |
| Lab 4 - RAG | loaders, chunking, Chroma, BM25, EnsembleRetriever, LCEL |
| Lab 5 - Advanced RAG | MMR and MultiQuery experiment |
| Lab 6 - RAG with LangGraph | START -> retrieve -> generate -> END |

Only small project-specific additions are made for building the one-time 36-source local policy dataset, attaching airline/category metadata and loading the resulting PDFs. The downstream RAG flow remains aligned to the class labs.

## 1. Initial Setup

In [1]:
# Same setup style as the in-class RAG notebooks
import os, json, re, getpass, warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from uuid import uuid4
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
load_dotenv(override=True)

True

In [2]:
# Check for Groq API key - same pattern used in class
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

### 1.1 Chat model - follows Lab 2 / Lab 4

In [3]:
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(
    model_name,
    model_provider="groq",
    temperature=0.1,  # low temperature from Lab 1 for more consistent factual answers
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x11ba623c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11ba62f90>, model_name='openai/gpt-oss-120b', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'))

### 1.2 Embedding model - follows Lab 4 / Lab 5

In [4]:
# Run once before this notebook if needed:
# !ollama pull nomic-embed-text

from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
embeddings_model

OllamaEmbeddings(model='nomic-embed-text', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## 2. Custom Dataset: 12 Airlines + DGCA/MoCA

In [5]:
# Resolve project paths whether the notebook is launched from /notebooks or project root
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SOURCE_FILE = PROJECT_ROOT / "data" / "airline_sources.csv"
EVAL_FILE = PROJECT_ROOT / "evaluation" / "evaluation_questions.csv"

# The original CSV is still supported, but Step 3 now embeds all 36 sources directly
# so the dataset-build cell does not depend on the CSV file being present.
if SOURCE_FILE.exists():
    sources_df = pd.read_csv(SOURCE_FILE)
    print("Original source manifest found:", SOURCE_FILE)
    print("Total curated sources:", len(sources_df))
    print("Airlines / regulator groups:", sources_df['airline'].nunique())
    display(sources_df[['airline','category','title','source_type']])
else:
    print("airline_sources.csv not found - no problem.")
    print("Step 3 contains the complete embedded 36-source manifest.")


Original source manifest found: /Users/milindnevrekar/generative-ai-pgp-ji-2026/PROJECT/GEN AI PROJECT/FlightWise_STAGE2_Final_Submission/FlightWise_GitHub_Repo/data/airline_sources.csv
Total curated sources: 36
Airlines / regulator groups: 13


,airline,category,title,source_type
0,DGCA / MoCA,regulatory,Passenger Charter of Rights,web
1,DGCA / MoCA,regulatory,"DGCA CAR - denied boarding, cancellation and d...",pdf
2,Air India,baggage,Checked Baggage Allowance,web
3,Air India,cancellation_refund,"Payment, Refund and Cancellation FAQ",web
4,Air India,conditions,Conditions of Carriage,pdf
5,IndiGo,conditions,Conditions of Carriage,web
6,IndiGo,baggage,Baggage Allowance,web
7,IndiGo,cancellation_refund,Flight Cancellation,web
8,Air India Express,baggage,Baggage and Fare Overview,web
9,Air India Express,cancellation_refund,Domestic Flights - Manage Booking,web


### Dataset design

Each source has metadata:
- `airline`
- `category`
- `title`
- `url`
- `source_type`
- `region`

This metadata is attached to every document before chunking. It allows FlightWise to prevent policy mixing between airlines.

## 3. Build the 36-Source Offline Dataset + Load It (One Cell)

This single cell contains the full 36-source manifest, downloads/normalizes the official sources into local PDFs, creates an offline ZIP, and loads the successful PDFs into `all_docs`. Re-running it uses cached valid PDFs whenever possible.


In [6]:
# ============================================================================
# STEP 3 - ONE CELL: DOWNLOAD + NORMALIZE + LOAD ALL 36 FLIGHTWISE SOURCES
# ============================================================================
# What this cell does:
# 1) Contains the complete 36-source FlightWise manifest inside the notebook.
# 2) Downloads official PDF files when the source is already a PDF.
# 3) For official web pages, saves the readable page content as a local PDF snapshot.
# 4) Uses browser impersonation first, normal requests second, and a reader fallback last.
# 5) Validates every PDF before keeping it.
# 6) Loads all successful local PDFs into `all_docs` for the normal RAG pipeline.
# 7) Creates `FlightWise_Official_Dataset.zip` so the corpus can be reused offline.
#
# This is DATA PREPARATION only. The RAG code after this cell still follows the
# instructor's PyPDFLoader -> splitter -> embeddings -> Chroma/retrievers flow.

import os
import re
import sys
import time
import json
import shutil
import zipfile
import textwrap
import subprocess
import importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd


# ---------------------------------------------------------------------------
# A. Install only the small helper packages needed for one-time dataset build
# ---------------------------------------------------------------------------
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f"Installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_package("curl_cffi", "curl_cffi")
ensure_package("bs4", "beautifulsoup4")
ensure_package("reportlab", "reportlab")
ensure_package("pypdf", "pypdf")
ensure_package("langchain_community", "langchain-community")

import requests
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests
from pypdf import PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from langchain_community.document_loaders import PyPDFLoader


# ---------------------------------------------------------------------------
# B. Complete custom dataset manifest - ALL 36 official sources
# ---------------------------------------------------------------------------
SOURCES = [{'source_id': 0,
  'airline': 'DGCA / MoCA',
  'category': 'regulatory',
  'title': 'Passenger Charter of Rights',
  'url': 'https://www.civilaviation.gov.in/ministry-documents/passenger-charter-of-rights',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Ministry of Civil Aviation passenger rights landing page'},
 {'source_id': 1,
  'airline': 'DGCA / MoCA',
  'category': 'regulatory',
  'title': 'DGCA CAR - denied boarding, cancellation and delays',
  'url': 'https://www.dgca.gov.in/digigov-portal/Upload?attachId=we1PSlOuQhYdHcwKKrm7ew%3D%3D&flag=iframeAttachView',
  'source_type': 'pdf',
  'region': 'India',
  'notes': 'DGCA Civil Aviation Requirement on passenger facilities during disruptions'},
 {'source_id': 2,
  'airline': 'Air India',
  'category': 'baggage',
  'title': 'Checked Baggage Allowance',
  'url': 'https://www.airindia.com/in/en/travel-information/baggage-guidelines/checked-baggage-allowance.html',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official checked baggage guidance'},
 {'source_id': 3,
  'airline': 'Air India',
  'category': 'cancellation_refund',
  'title': 'Payment, Refund and Cancellation FAQ',
  'url': 'https://www.airindia.com/in/en/frequently-asked-questions/payment-refund-cancellations.html',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official refund and cancellation FAQ'},
 {'source_id': 4,
  'airline': 'Air India',
  'category': 'conditions',
  'title': 'Conditions of Carriage',
  'url': 'https://www.airindia.com/content/dam/air-india/pdfs/conditions-of-carriage-for-passengers-and-baggage.pdf',
  'source_type': 'pdf',
  'region': 'India',
  'notes': 'Official conditions of carriage PDF'},
 {'source_id': 5,
  'airline': 'IndiGo',
  'category': 'conditions',
  'title': 'Conditions of Carriage',
  'url': 'https://www.goindigo.in/information/conditions-of-carriage.html',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official conditions covering cancellation, refund, passenger rights and baggage'},
 {'source_id': 6,
  'airline': 'IndiGo',
  'category': 'baggage',
  'title': 'Baggage Allowance',
  'url': 'https://www.goindigo.in/baggage/baggage-allowance.html',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official baggage allowance page'},
 {'source_id': 7,
  'airline': 'IndiGo',
  'category': 'cancellation_refund',
  'title': 'Flight Cancellation',
  'url': 'https://www.goindigo.in/cancellation.html',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official cancellation page'},
 {'source_id': 8,
  'airline': 'Air India Express',
  'category': 'baggage',
  'title': 'Baggage and Fare Overview',
  'url': 'https://flights.airindiaexpress.com/en-in/',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official site with baggage allowance and booking management information'},
 {'source_id': 9,
  'airline': 'Air India Express',
  'category': 'cancellation_refund',
  'title': 'Domestic Flights - Manage Booking',
  'url': 'https://flights.airindiaexpress.com/en-in/domestic-flights',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official page with cancellation, modification and status guidance'},
 {'source_id': 10,
  'airline': 'Akasa Air',
  'category': 'baggage',
  'title': 'Excess Baggage',
  'url': 'https://www.akasaair.com/add-ons/excess-baggage',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official baggage allowances and excess baggage terms'},
 {'source_id': 11,
  'airline': 'Akasa Air',
  'category': 'check_in',
  'title': 'Web Check-In',
  'url': 'https://www.akasaair.com/check-in/web-check-in',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official check-in rules'},
 {'source_id': 12,
  'airline': 'Akasa Air',
  'category': 'cancellation_refund',
  'title': 'Flexi Cover',
  'url': 'https://www.akasaair.com/add-ons/flexi-cover',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official cancellation/amendment cover information'},
 {'source_id': 13,
  'airline': 'SpiceJet',
  'category': 'conditions',
  'title': 'Terms and Conditions / Conditions of Carriage',
  'url': 'https://corporate.spicejet.com/Tnc.aspx',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official conditions of carriage'},
 {'source_id': 14,
  'airline': 'SpiceJet',
  'category': 'passenger_rights',
  'title': 'Passenger Rights',
  'url': 'https://corporate.spicejet.com/PassengerRights.aspx',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official passenger rights and disruption/refund information'},
 {'source_id': 15,
  'airline': 'SpiceJet',
  'category': 'baggage',
  'title': 'Baggage',
  'url': 'https://corporate.spicejet.com/Baggage.aspx',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official baggage policy'},
 {'source_id': 16,
  'airline': 'Alliance Air',
  'category': 'baggage',
  'title': 'Baggage Information',
  'url': 'https://plone.allianceair.in/allianceair/en/pages/baggageinfo',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official baggage information hub'},
 {'source_id': 17,
  'airline': 'Alliance Air',
  'category': 'baggage',
  'title': 'Excess Baggage',
  'url': 'https://plone.allianceair.in/allianceair/en/pages/excess-baggage',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official excess baggage charges'},
 {'source_id': 18,
  'airline': 'Alliance Air',
  'category': 'cancellation_refund',
  'title': 'Change / Cancel Reservation FAQ',
  'url': 'https://plone.allianceair.in/allianceair/en/faq/managebooking/how-can-i-change-cancel-my-reservation',
  'source_type': 'web',
  'region': 'India',
  'notes': 'Official change/cancel booking FAQ'},
 {'source_id': 19,
  'airline': 'Emirates',
  'category': 'baggage',
  'title': 'Checked Baggage',
  'url': 'https://www.emirates.com/english/before-you-fly/baggage/checked-baggage/',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official checked baggage page'},
 {'source_id': 20,
  'airline': 'Emirates',
  'category': 'cancellation_refund',
  'title': 'Cancellation Notice',
  'url': 'https://www.emirates.com/english/before-you-fly/travel/rules-and-notices/cancellation-notice/',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official cancellation rights notice'},
 {'source_id': 21,
  'airline': 'Emirates',
  'category': 'cancellation_refund',
  'title': 'Refund Request',
  'url': 'https://www.emirates.com/english/help/forms/refund-request/',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official refund request information'},
 {'source_id': 22,
  'airline': 'Qatar Airways',
  'category': 'conditions',
  'title': 'Conditions of Carriage',
  'url': 'https://www.qatarairways.com/en/legal/conditions-of-carriage.html',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official conditions of carriage'},
 {'source_id': 23,
  'airline': 'Qatar Airways',
  'category': 'baggage',
  'title': 'Optional Services and Fees',
  'url': 'https://www.qatarairways.com/en/optional-services-and-fees.html',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official checked baggage and service fee details'},
 {'source_id': 24,
  'airline': 'Etihad Airways',
  'category': 'conditions',
  'title': 'General Conditions of Carriage',
  'url': 'https://www.etihad.com/en/legal/conditions-of-carriage',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official conditions of carriage'},
 {'source_id': 25,
  'airline': 'Etihad Airways',
  'category': 'baggage',
  'title': 'Baggage Information',
  'url': 'https://www.etihad.com/en/help/baggage-information',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official baggage calculator/policy information'},
 {'source_id': 26,
  'airline': 'Etihad Airways',
  'category': 'cancellation_refund',
  'title': 'Fare Types and Changes',
  'url': 'https://www.etihad.com/en/help/faq/fares',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official fare rules with changes/cancellations'},
 {'source_id': 27,
  'airline': 'Singapore Airlines',
  'category': 'conditions',
  'title': 'Conditions of Carriage',
  'url': 'https://www.singaporeair.com/en_UK/global_footer/conditions-carriage/',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official general conditions of carriage'},
 {'source_id': 28,
  'airline': 'Singapore Airlines',
  'category': 'baggage',
  'title': 'Checked Baggage',
  'url': 'https://www.singaporeair.com/en_UK/za/travel-info/baggage/checked-baggage/',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official checked baggage rules'},
 {'source_id': 29,
  'airline': 'Singapore Airlines',
  'category': 'baggage',
  'title': 'Delayed Baggage Report',
  'url': 'https://www.singaporeair.com/baggage-service/report-delayed-bag',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official delayed baggage reporting requirements'},
 {'source_id': 30,
  'airline': 'Lufthansa',
  'category': 'conditions',
  'title': 'General Conditions of Carriage',
  'url': 'https://www.lufthansa.com/pl/en/terms-and-conditions-lh',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official conditions of carriage'},
 {'source_id': 31,
  'airline': 'Lufthansa',
  'category': 'cancellation_refund',
  'title': 'Refunds',
  'url': 'https://www.lufthansa.com/us/en/refund',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official refund rules'},
 {'source_id': 32,
  'airline': 'Lufthansa',
  'category': 'baggage',
  'title': 'Baggage and Other Fees',
  'url': 'https://www.lufthansa.com/us/en/baggage-and-other-fees',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official baggage rules and fees'},
 {'source_id': 33,
  'airline': 'British Airways',
  'category': 'conditions',
  'title': 'General Conditions of Carriage',
  'url': 'https://www.britishairways.com/content/information/legal/conditions-of-carriage',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official conditions of carriage'},
 {'source_id': 34,
  'airline': 'British Airways',
  'category': 'cancellation_refund',
  'title': 'Cancelled Flight FAQ',
  'url': 'https://www.britishairways.com/content/information/help/faq/disruption/cancellations/cancelled-flight',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official cancellation/rebooking/refund guidance'},
 {'source_id': 35,
  'airline': 'British Airways',
  'category': 'baggage',
  'title': 'Baggage FAQ',
  'url': 'https://www.britishairways.com/content/information/help/faqs/day-of-travel',
  'source_type': 'web',
  'region': 'International',
  'notes': 'Official baggage FAQ and travel-day information'}]

assert len(SOURCES) == 36, "FlightWise manifest should contain exactly 36 sources."
sources_df = pd.DataFrame(SOURCES)

print("FlightWise sources in this notebook:", len(sources_df))
print("Airlines / regulator groups:", sources_df['airline'].nunique())
display(sources_df[['source_id', 'airline', 'category', 'title', 'source_type']])


# ---------------------------------------------------------------------------
# C. Local folders
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

DATA_ROOT = PROJECT_ROOT / "data"
POLICY_DIR = DATA_ROOT / "policies"
POLICY_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_MANIFEST = DATA_ROOT / "local_source_manifest.csv"
DATASET_ZIP = PROJECT_ROOT / "FlightWise_Official_Dataset.zip"

# Preserve the original evaluation path used later in the notebook.
EVAL_FILE = PROJECT_ROOT / "evaluation" / "evaluation_questions.csv"

print("\nLocal policy folder:", POLICY_DIR)


# ---------------------------------------------------------------------------
# D. Download settings
# ---------------------------------------------------------------------------
REQUEST_TIMEOUT = 18
READER_TIMEOUT = 30
MAX_WORKERS = 6                 # parallel sources -> much faster than one-by-one
MIN_TEXT_LENGTH = 500
MAX_SNAPSHOT_CHARS = 250000     # avoids accidentally saving gigantic web pages

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


# ---------------------------------------------------------------------------
# E. Helper functions
# ---------------------------------------------------------------------------
def safe_name(value):
    value = str(value).strip().replace("/", "_")
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_.")
    return value or "source"


def pdf_signature(data):
    return isinstance(data, (bytes, bytearray)) and data[:5] == b"%PDF-"


def validate_pdf(path):
    try:
        reader = PdfReader(str(path))
        return len(reader.pages) > 0
    except Exception:
        return False


def normalize_text_for_pdf(text):
    replacements = {
        "\u00a0": " ",
        "\u2018": "'", "\u2019": "'",
        "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-",
        "\u2022": "*",
        "₹": "INR ", "€": "EUR ", "£": "GBP ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    # Helvetica in ReportLab is most reliable with Latin-1; remove unsupported glyphs.
    return text.encode("latin-1", errors="ignore").decode("latin-1")


def html_to_clean_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript", "svg"]):
        tag.decompose()

    container = soup.find("main") or soup.find("article") or soup.body or soup
    text = "\n".join(s.strip() for s in container.stripped_strings if s.strip())
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def looks_like_block_page(text):
    t = text.lower()[:5000]
    bad_phrases = [
        "access denied", "request blocked", "forbidden", "captcha",
        "security check", "verify you are human", "enable javascript to continue",
        "checking your browser", "temporarily unavailable"
    ]
    return any(p in t for p in bad_phrases)


def write_text_snapshot_pdf(text, output_path, title, official_url):
    text = normalize_text_for_pdf(text[:MAX_SNAPSHOT_CHARS])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    c = canvas.Canvas(str(output_path), pagesize=A4)
    width, height = A4
    margin = 42
    y = height - margin

    def new_page():
        nonlocal y
        c.showPage()
        y = height - margin

    c.setFont("Helvetica-Bold", 12)
    for line in textwrap.wrap(normalize_text_for_pdf(title), width=85):
        c.drawString(margin, y, line)
        y -= 16

    c.setFont("Helvetica", 7)
    for line in textwrap.wrap("Official source: " + official_url, width=130):
        c.drawString(margin, y, line)
        y -= 10

    y -= 8
    c.setFont("Helvetica", 8.5)

    for raw_line in text.splitlines():
        raw_line = raw_line.strip()
        if not raw_line:
            y -= 5
            continue

        wrapped = textwrap.wrap(raw_line, width=105, break_long_words=True, break_on_hyphens=True) or [""]
        for line in wrapped:
            if y < margin + 20:
                new_page()
                c.setFont("Helvetica", 8.5)
            c.drawString(margin, y, line)
            y -= 10.5

    c.save()


def fetch_direct(url):
    errors = []

    # Method 1: curl_cffi impersonates a real Chrome TLS/browser fingerprint.
    try:
        r = curl_requests.get(
            url,
            headers=HEADERS,
            impersonate="chrome",
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )
        if r.status_code < 400 and len(r.content) > 100:
            return r.content, r.headers.get("content-type", ""), str(r.url), "curl_cffi"
        errors.append(f"curl_cffi HTTP {r.status_code}")
    except Exception as e:
        errors.append(f"curl_cffi {type(e).__name__}: {str(e)[:100]}")

    # Method 2: normal requests fallback.
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        if r.status_code < 400 and len(r.content) > 100:
            return r.content, r.headers.get("content-type", ""), str(r.url), "requests"
        errors.append(f"requests HTTP {r.status_code}")
    except Exception as e:
        errors.append(f"requests {type(e).__name__}: {str(e)[:100]}")

    raise RuntimeError(" | ".join(errors))


def fetch_reader_text(url):
    # Final fallback. It reads the PUBLIC official URL and returns extracted text.
    # The original airline/regulator URL remains the source of record in metadata.
    reader_url = "https://r.jina.ai/" + url
    r = requests.get(
        reader_url,
        headers={"User-Agent": HEADERS["User-Agent"], "Accept": "text/plain"},
        timeout=READER_TIMEOUT,
        allow_redirects=True,
    )
    r.raise_for_status()
    text = r.text.strip()
    if len(text) < MIN_TEXT_LENGTH or looks_like_block_page(text):
        raise ValueError("Reader fallback returned too little/blocked text")
    return text


def output_path_for(source):
    airline_folder = safe_name(source['airline'])
    category_folder = safe_name(source['category'])
    filename = f"{int(source['source_id']):02d}_{safe_name(source['title'])}.pdf"
    return POLICY_DIR / airline_folder / category_folder / filename


def process_source(source):
    output_path = output_path_for(source)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    base_result = {
        "source_id": source['source_id'],
        "airline": source['airline'],
        "category": source['category'],
        "title": source['title'],
        "official_url": source['url'],
        "source_type": source['source_type'],
        "region": source['region'],
        "local_file": str(output_path),
    }

    # Re-running the cell should reuse valid local files rather than re-download them.
    if output_path.exists() and validate_pdf(output_path):
        return {**base_result, "status": "cached", "method": "local_cache", "error": ""}

    if output_path.exists():
        output_path.unlink(missing_ok=True)

    errors = []

    # First try the original official URL directly.
    try:
        content, content_type, final_url, method = fetch_direct(source['url'])

        # A real PDF: keep the original official bytes.
        if pdf_signature(content):
            output_path.write_bytes(content)
            if validate_pdf(output_path):
                return {**base_result, "status": "downloaded_pdf", "method": method, "error": ""}
            output_path.unlink(missing_ok=True)
            errors.append("Direct response looked like PDF but validation failed")

        # A web page: normalize official page text into a local PDF snapshot.
        else:
            html = content.decode("utf-8", errors="ignore")
            text = html_to_clean_text(html)
            if len(text) >= MIN_TEXT_LENGTH and not looks_like_block_page(text):
                write_text_snapshot_pdf(text, output_path, source['title'], source['url'])
                if validate_pdf(output_path):
                    return {**base_result, "status": "web_snapshot_pdf", "method": method, "error": ""}
                output_path.unlink(missing_ok=True)
            else:
                errors.append("Direct web response contained too little/blocked readable text")

    except Exception as e:
        errors.append(f"Direct: {type(e).__name__}: {str(e)[:180]}")

    # Final fallback for public pages/PDFs that reject direct programmatic downloads.
    try:
        text = fetch_reader_text(source['url'])
        write_text_snapshot_pdf(text, output_path, source['title'], source['url'])
        if validate_pdf(output_path):
            return {**base_result, "status": "reader_snapshot_pdf", "method": "reader_fallback", "error": ""}
        output_path.unlink(missing_ok=True)
        errors.append("Reader snapshot PDF validation failed")
    except Exception as e:
        errors.append(f"Reader: {type(e).__name__}: {str(e)[:180]}")

    return {**base_result, "status": "failed", "method": "none", "error": " | ".join(errors)}


# ---------------------------------------------------------------------------
# F. Download all 36 sources IN PARALLEL
# ---------------------------------------------------------------------------
print("\nStarting one-time FlightWise dataset build...")
print(f"Using up to {MAX_WORKERS} parallel workers.\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {executor.submit(process_source, s): s for s in SOURCES}

    completed = 0
    for future in as_completed(future_map):
        source = future_map[future]
        completed += 1
        try:
            result = future.result()
        except Exception as e:
            result = {
                "source_id": source['source_id'],
                "airline": source['airline'],
                "category": source['category'],
                "title": source['title'],
                "official_url": source['url'],
                "source_type": source['source_type'],
                "region": source['region'],
                "local_file": str(output_path_for(source)),
                "status": "failed",
                "method": "none",
                "error": f"Worker {type(e).__name__}: {str(e)[:180]}",
            }

        results.append(result)
        print(
            f"[{completed:02d}/36] {result['status']:<20} | "
            f"{result['airline']} | {result['title']}"
        )


# ---------------------------------------------------------------------------
# G. Save local source manifest and show download report
# ---------------------------------------------------------------------------
local_manifest_df = pd.DataFrame(results).sort_values('source_id').reset_index(drop=True)
local_manifest_df.to_csv(LOCAL_MANIFEST, index=False)

print("\n" + "=" * 90)
print("DOWNLOAD SUMMARY")
print("=" * 90)
print(local_manifest_df['status'].value_counts(dropna=False).to_string())

print("\nCoverage by airline/regulator group:")
coverage = (
    local_manifest_df.assign(success=local_manifest_df['status'] != 'failed')
    .groupby('airline')
    .agg(total_sources=('source_id', 'count'), successful_sources=('success', 'sum'))
    .reset_index()
)
display(coverage)

failed_downloads_df = local_manifest_df[local_manifest_df['status'] == 'failed']
if not failed_downloads_df.empty:
    print("\nSources still requiring manual download/save-as-PDF:")
    display(failed_downloads_df[['source_id', 'airline', 'title', 'official_url', 'error']])
else:
    print("\nAll 36 sources were converted into valid local PDFs.")


# ---------------------------------------------------------------------------
# H. Load every successful local PDF into LangChain `all_docs`
# ---------------------------------------------------------------------------
all_docs = []
failed_local_loads = []

for _, item in local_manifest_df[local_manifest_df['status'] != 'failed'].iterrows():
    pdf_path = Path(item['local_file'])

    try:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()

        for doc in docs:
            doc.metadata.update({
                'source_id': int(item['source_id']),
                'airline': item['airline'],
                'category': item['category'],
                'title': item['title'],
                'source_url': item['official_url'],
                'region': item['region'],
                'local_file': str(pdf_path),
                'dataset_status': item['status'],
            })

        all_docs.extend(docs)

    except Exception as e:
        failed_local_loads.append({
            'source_id': int(item['source_id']),
            'airline': item['airline'],
            'file': str(pdf_path),
            'error': f"{type(e).__name__}: {str(e)[:180]}",
        })

print("\n" + "=" * 90)
print("LOCAL RAG CORPUS READY")
print("=" * 90)
print("Successful source PDFs available:", int((local_manifest_df['status'] != 'failed').sum()))
print("LangChain PDF pages loaded into all_docs:", len(all_docs))
print("Local PDF load failures:", len(failed_local_loads))

if failed_local_loads:
    display(pd.DataFrame(failed_local_loads))


# ---------------------------------------------------------------------------
# I. ZIP the downloaded custom dataset for reuse/offline demo
# ---------------------------------------------------------------------------
if DATASET_ZIP.exists():
    DATASET_ZIP.unlink()

with zipfile.ZipFile(DATASET_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pdf in POLICY_DIR.rglob('*.pdf'):
        zf.write(pdf, pdf.relative_to(PROJECT_ROOT))
    if LOCAL_MANIFEST.exists():
        zf.write(LOCAL_MANIFEST, LOCAL_MANIFEST.relative_to(PROJECT_ROOT))

print("\nOffline dataset ZIP created:")
print(DATASET_ZIP)
print("\nNext step: run Step 4 (RecursiveCharacterTextSplitter).")

FlightWise sources in this notebook: 36
Airlines / regulator groups: 13


,source_id,airline,category,title,source_type
0,0,DGCA / MoCA,regulatory,Passenger Charter of Rights,web
1,1,DGCA / MoCA,regulatory,"DGCA CAR - denied boarding, cancellation and d...",pdf
2,2,Air India,baggage,Checked Baggage Allowance,web
3,3,Air India,cancellation_refund,"Payment, Refund and Cancellation FAQ",web
4,4,Air India,conditions,Conditions of Carriage,pdf
5,5,IndiGo,conditions,Conditions of Carriage,web
6,6,IndiGo,baggage,Baggage Allowance,web
7,7,IndiGo,cancellation_refund,Flight Cancellation,web
8,8,Air India Express,baggage,Baggage and Fare Overview,web
9,9,Air India Express,cancellation_refund,Domestic Flights - Manage Booking,web



Local policy folder: /Users/milindnevrekar/generative-ai-pgp-ji-2026/PROJECT/GEN AI PROJECT/FlightWise_STAGE2_Final_Submission/FlightWise_GitHub_Repo/data/policies

Starting one-time FlightWise dataset build...
Using up to 6 parallel workers.

[01/36] cached               | Air India | Checked Baggage Allowance
[02/36] cached               | DGCA / MoCA | DGCA CAR - denied boarding, cancellation and delays
[03/36] cached               | IndiGo | Conditions of Carriage
[04/36] cached               | Air India | Payment, Refund and Cancellation FAQ
[05/36] cached               | Air India | Conditions of Carriage
[06/36] cached               | IndiGo | Flight Cancellation
[07/36] cached               | Akasa Air | Excess Baggage
[08/36] cached               | Air India Express | Baggage and Fare Overview
[09/36] cached               | Akasa Air | Web Check-In
[10/36] cached               | Akasa Air | Flexi Cover
[11/36] cached               | SpiceJet | Terms and Conditions / Condition

,airline,total_sources,successful_sources
0,Air India,3,3
1,Air India Express,2,1
2,Akasa Air,3,3
3,Alliance Air,3,0
4,British Airways,3,3
5,DGCA / MoCA,2,1
6,Emirates,3,3
7,Etihad Airways,3,3
8,IndiGo,3,2
9,Lufthansa,3,3



Sources still requiring manual download/save-as-PDF:


,source_id,airline,title,official_url,error
0,0,DGCA / MoCA,Passenger Charter of Rights,https://www.civilaviation.gov.in/ministry-docu...,Direct web response contained too little/block...
6,6,IndiGo,Baggage Allowance,https://www.goindigo.in/baggage/baggage-allowa...,Direct web response contained too little/block...
9,9,Air India Express,Domestic Flights - Manage Booking,https://flights.airindiaexpress.com/en-in/dome...,Direct web response contained too little/block...
16,16,Alliance Air,Baggage Information,https://plone.allianceair.in/allianceair/en/pa...,Direct web response contained too little/block...
17,17,Alliance Air,Excess Baggage,https://plone.allianceair.in/allianceair/en/pa...,Direct web response contained too little/block...
18,18,Alliance Air,Change / Cancel Reservation FAQ,https://plone.allianceair.in/allianceair/en/fa...,Direct web response contained too little/block...



LOCAL RAG CORPUS READY
Successful source PDFs available: 30
LangChain PDF pages loaded into all_docs: 313
Local PDF load failures: 0

Offline dataset ZIP created:
/Users/milindnevrekar/generative-ai-pgp-ji-2026/PROJECT/GEN AI PROJECT/FlightWise_STAGE2_Final_Submission/FlightWise_GitHub_Repo/FlightWise_Official_Dataset.zip

Next step: run Step 4 (RecursiveCharacterTextSplitter).


**Important for the demo:** Airline websites can change or block automated requests. A few failed sources do not stop the project because the custom corpus contains multiple sources. Re-check the failed-source table before the final demo.

## 4. Split Documents - same baseline as Lab 4

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # same baseline used in class
    chunk_overlap=200,     # same baseline used in class
    add_start_index=True,
)

all_splits = text_splitter.split_documents(all_docs)
print(f"Split {len(all_docs)} loaded documents into {len(all_splits)} chunks.")

Split 313 loaded documents into 1491 chunks.


In [8]:
# Inspect metadata attached to a chunk
if all_splits:
    print(all_splits[0].metadata)
    print(all_splits[0].page_content[:500])

{'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-01-25T16:48:47+05:30', 'author': 'acer', 'moddate': '2023-01-25T16:48:47+05:30', 'source': '/Users/milindnevrekar/generative-ai-pgp-ji-2026/PROJECT/GEN AI PROJECT/FlightWise_STAGE2_Final_Submission/FlightWise_GitHub_Repo/data/policies/DGCA_MoCA/regulatory/01_DGCA_CAR_-_denied_boarding_cancellation_and_delays.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_id': 1, 'airline': 'DGCA / MoCA', 'category': 'regulatory', 'title': 'DGCA CAR - denied boarding, cancellation and delays', 'source_url': 'https://www.dgca.gov.in/digigov-portal/Upload?attachId=we1PSlOuQhYdHcwKKrm7ew%3D%3D&flag=iframeAttachView', 'region': 'India', 'local_file': '/Users/milindnevrekar/generative-ai-pgp-ji-2026/PROJECT/GEN AI PROJECT/FlightWise_STAGE2_Final_Submission/FlightWise_GitHub_Repo/data/policies/DGCA_MoCA/regulatory/01_DGCA_CAR_-_denied_boarding_cancellation_and_delays.pdf', 'dataset_status': 'cached', '

## 5. Store Chunks in Chroma - follows Lab 4 / Lab 5

In [9]:
print(type(embeddings_model))
print(embeddings_model.model)


test_embedding = embeddings_model.embed_query(
    "What is the baggage allowance?"
)
print(len(test_embedding))


chunk_lengths = [
    len(doc.page_content)
    for doc in all_splits
]

print("Number of chunks:", len(all_splits))
print("Maximum:", max(chunk_lengths))
print("Average:", sum(chunk_lengths) / len(chunk_lengths))

<class 'langchain_ollama.embeddings.OllamaEmbeddings'>
nomic-embed-text
768
Number of chunks: 1491
Maximum: 1000
Average: 875.3863179074447


In [10]:
for i, doc in enumerate(all_splits):
    try:
        embeddings_model.embed_query(doc.page_content)

        if (i + 1) % 100 == 0:
            print(f"Successfully embedded: {i + 1}")

    except Exception as e:
        print(f"FAILED at chunk {i}")
        print("Chunk length:", len(doc.page_content))
        print("Chunk preview:", repr(doc.page_content[:500]))
        print("Error:", e)
        break
else:
    print("All chunks embedded successfully!")
    
    


Successfully embedded: 100
Successfully embedded: 200
Successfully embedded: 300
Successfully embedded: 400
Successfully embedded: 500
Successfully embedded: 600
Successfully embedded: 700
Successfully embedded: 800
Successfully embedded: 900
Successfully embedded: 1000
Successfully embedded: 1100
Successfully embedded: 1200
Successfully embedded: 1300
Successfully embedded: 1400
All chunks embedded successfully!


In [11]:
#print("Existing documents:", vector_store_chroma._collection.count())

In [12]:

from langchain_chroma import Chroma
from uuid import uuid4

PERSIST_DIR = str(PROJECT_ROOT / "chroma_flightwise_db")

vector_store_chroma = Chroma(
    collection_name="flightwise-rag",
    embedding_function=embeddings_model,
    persist_directory=PERSIST_DIR,
)

# Check existing documents
existing_count = vector_store_chroma._collection.count()

print("Existing documents:", existing_count)

if existing_count > 0:

    print("Collection already contains documents.")
    print("No new documents added.")

else:

    # Generate unique IDs for all chunks
    uuids = [
        str(uuid4())
        for _ in range(len(all_splits))
    ]

    # Batch size
    BATCH_SIZE = 32

    # Embed and store chunks in batches
    for start in range(0, len(all_splits), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(all_splits))

        batch_documents = all_splits[start:end]
        batch_ids = uuids[start:end]

        # Extract text
        batch_texts = [
            doc.page_content
            for doc in batch_documents
        ]

        # Extract metadata
        batch_metadatas = [
            doc.metadata
            for doc in batch_documents
        ]

        print(f"Embedding chunks {start} to {end - 1}...")

        # STEP 1: Generate embeddings
        batch_embeddings = embeddings_model.embed_documents(
            batch_texts
        )

        print(f"Generated {len(batch_embeddings)} embeddings")

        # STEP 2: Store precomputed embeddings in ChromaDB
        vector_store_chroma._collection.add(
            ids=batch_ids,
            documents=batch_texts,
            embeddings=batch_embeddings,
            metadatas=batch_metadatas
        )

        print(f"Stored {end} / {len(all_splits)} chunks")

    print("Indexing completed!")
    print("Total documents:", vector_store_chroma._collection.count())

Existing documents: 1491
Collection already contains documents.
No new documents added.


## 6. Retrieval Methods from Class

### 6.1 Similarity retrieval

In [13]:
def get_similarity_retriever(airline, k=3):
    return vector_store_chroma.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k, "filter": {"airline": airline}}
    )

### 6.2 MMR retrieval

In [14]:
def get_mmr_retriever(airline, k=3):
    return vector_store_chroma.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k, "filter": {"airline": airline}}
    )

### 6.3 Hybrid retrieval: BM25 + Chroma - follows Lab 4

In [15]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

def get_airline_splits(airline):
    return [doc for doc in all_splits if doc.metadata.get('airline') == airline]

def get_hybrid_retriever(airline, k=3):
    selected_docs = get_airline_splits(airline)
    if not selected_docs:
        raise ValueError(f"No indexed chunks found for {airline}")

    bm25_retriever = BM25Retriever.from_documents(documents=selected_docs)
    bm25_retriever.k = k

    chroma_retriever = vector_store_chroma.as_retriever(
        search_kwargs={"k": k, "filter": {"airline": airline}}
    )

    # Same weights as the class example
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, chroma_retriever],
        weights=[0.7, 0.3]
    )
    return ensemble_retriever

### 6.4 Inspect retrieval before generation

In [16]:
selected_airline = "IndiGo"
question = "My suitcase did not arrive. What should I do?"

hybrid_retriever = get_hybrid_retriever(selected_airline)
retrieved_docs = hybrid_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Result {i} ---")
    print("Airline:", doc.metadata.get('airline'))
    print("Category:", doc.metadata.get('category'))
    print("Title:", doc.metadata.get('title'))
    print(doc.page_content[:500], "\n")


--- Result 1 ---
Airline: IndiGo
Category: cancellation_refund
Title: Flight Cancellation
transportation to and fro the airport and hotel and dinner.
Till when can I cancel/ refund my booking online?
On domestic flights you can cancel/ refund till 3 hours prior to flight departure
On international flights you can cancel/ refund till 4 hours prior to flight departure
What is the procedure to cancel my flight?
The customers booking can be cancelled/ changed before the scheduled time of departure by contacting
IndiGo's Call Centre, at IndiGo's Airport Counters, on Indigo's website and t 

--- Result 2 ---
Airline: IndiGo
Category: conditions
Title: Conditions of Carriage
time). However, IndiGo endeavours to give priority to the following categories of Customers:
A Child;
A safety assistant;
A pregnant woman;
Customers with one or more sensory impairments such as a visual, hearing or speech impairment;
Customers using or requiring wheelchairs, crutches, braces, other prosthetic devices or 

## 7. Prompt - follows Lab 4 ChatPromptTemplate

In [17]:
from langchain_core.prompts import ChatPromptTemplate

system_message_template = """You are FlightWise, an airline passenger policy assistant.

Use ONLY the retrieved context supplied to you.
Do not invent airline rules, fees, entitlements, time limits or compensation.
If the context does not contain enough information, clearly say that sufficient information was not found in the retrieved policy sources.
Keep the answer practical and easy for a passenger to understand.
Always mention the source document titles used.

Selected airline: {airline}

Retrieved context:
{context}
"""

flightwise_prompt = ChatPromptTemplate([
    ("system", system_message_template),
    ("human", "Passenger question: {question}"),
])

## 8. Structured Output - follows Lab 3

In [18]:
from pydantic import BaseModel, Field

class PassengerSupportResponse(BaseModel):
    issue_type: str = Field(description="Short label for the passenger issue")
    policy_summary: str = Field(description="Answer based only on retrieved policy context")
    recommended_steps: list[str] = Field(description="Practical next steps supported by retrieved context")
    source_documents: list[str] = Field(description="Titles of the retrieved policy sources used")
    information_gap: bool = Field(description="True if the retrieved context is insufficient")

structured_llm = llm.with_structured_output(PassengerSupportResponse)

## 9. LCEL RAG Chain - same idea as Lab 4

In [19]:
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(
            f"SOURCE TITLE: {doc.metadata.get('title')}\n"
            f"AIRLINE: {doc.metadata.get('airline')}\n"
            f"CATEGORY: {doc.metadata.get('category')}\n"
            f"URL: {doc.metadata.get('source_url')}\n"
            f"CONTENT: {doc.page_content}"
        )
    return "\n\n".join(formatted)


def ask_flightwise_lcel(airline, question):
    retriever = get_hybrid_retriever(airline)
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = flightwise_prompt | structured_llm
    answer = chain.invoke({
        "airline": airline,
        "question": question,
        "context": context,
    })
    return answer, docs


In [20]:
answer, evidence = ask_flightwise_lcel(
    "Air India",
    "If Air India cancels my flight, how can I request a refund?"
)

print(answer.model_dump())

{'issue_type': 'refund_request_flight_cancellation', 'policy_summary': 'If Air India cancels your flight, you can request a refund through the airline’s website or mobile app by using the ‘manage booking’ section. Alternatively, you may choose to reschedule via the self‑service re‑accommodation option on the website or by following the reschedule link sent to your registered email.', 'recommended_steps': ['Log in to the Air India website or open the Air India mobile app.', 'Navigate to the ‘Manage Booking’ or ‘My Trips’ section.', 'Select the cancelled flight and choose the ‘Refund’ option to submit a refund request.', 'If you prefer to travel on a later date, use the ‘Self‑service Re‑accommodation’ feature on the website or click the reschedule link provided in the cancellation email.', 'After submitting, monitor the status via the ‘Status Check on Pending Refund’ on the customer support portal.'], 'source_documents': ['Payment, Refund and Cancellation FAQ'], 'information_gap': False}

## 10. Advanced RAG Experiment - MultiQuery from Lab 5

In [21]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# Example experiment on one selected airline.
# MultiQuery can create alternative wordings when the passenger uses informal language.
base_retriever = get_similarity_retriever("IndiGo", k=2)

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

# Uncomment to inspect generated retrieval results:
# mq_docs = mq_retriever.invoke("My suitcase never came")
# [(d.metadata.get('title'), d.metadata.get('category')) for d in mq_docs]

## 11. Final Workflow with LangGraph - follows Lab 6

In [22]:
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph, END

class State(TypedDict):
    airline: str
    question: str
    context: List[Document]
    answer: PassengerSupportResponse


def retrieve(state: State):
    retriever = get_hybrid_retriever(state['airline'])
    retrieved_docs = retriever.invoke(state['question'])
    return {"context": retrieved_docs}


def generate(state: State):
    context = format_docs(state['context'])
    response = (flightwise_prompt | structured_llm).invoke({
        "airline": state['airline'],
        "question": state['question'],
        "context": context,
    })
    return {"answer": response}


graph_builder = StateGraph(State)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

In [23]:
result = graph.invoke({
    "airline": "Emirates",
    "question": "How is checked baggage allowance calculated?"
})

print(result['answer'].model_dump())

{'issue_type': 'Checked baggage allowance calculation', 'policy_summary': 'Emirates calculates your free checked baggage allowance using either a weight concept or a piece concept. The weight concept, which applies to most routes, gives you a total weight limit that depends on your travel class and fare (e.g., Premium Economy 35\u202fkg, Business 40\u202fkg, First 50\u202fkg; Economy varies by fare). The piece concept applies only on flights to and from the Americas and Africa. Under the piece concept you are allowed a specific number of bags, each with its own weight limit (e.g., Economy Special – 1\u202f×\u202f23\u202fkg; Economy Saver – 1\u202f×\u202f23\u202fkg; Economy Flex/Flex\u202fPlus – 2\u202f×\u202f23\u202fkg; Premium Economy – 2\u202f×\u202f23\u202fkg; Business/First – 2\u202f×\u202f32\u202fkg). The dimensions of each bag must not exceed 150\u202fcm (length\u202f+\u202fwidth\u202f+\u202fheight) for the piece concept or 203\u202fcm for the weight concept.', 'recommended_steps

## 12. Airline Comparison Feature

In [24]:
def compare_airlines(airline_1, airline_2, question):
    answer_1, docs_1 = ask_flightwise_lcel(airline_1, question)
    answer_2, docs_2 = ask_flightwise_lcel(airline_2, question)

    comparison = pd.DataFrame({
        "Airline": [airline_1, airline_2],
        "Policy summary": [answer_1.policy_summary, answer_2.policy_summary],
        "Sources": [", ".join(answer_1.source_documents), ", ".join(answer_2.source_documents)]
    })
    return comparison

# Example:
# display(compare_airlines("Air India", "IndiGo", "What does the policy say about checked baggage?"))

## 13. Retrieval Evaluation - Similarity vs MMR vs Hybrid

In [25]:
eval_df = pd.read_csv(EVAL_FILE)
print("Evaluation questions:", len(eval_df))
display(eval_df.head())

Evaluation questions: 38


,question_id,airline,expected_category,question
0,Q01,Air India,baggage,What checked baggage allowance applies on Air ...
1,Q02,Air India,baggage,What is the maximum weight allowed for a singl...
2,Q03,Air India,cancellation_refund,"If Air India cancels my flight, how can I requ..."
3,Q04,IndiGo,baggage,How much hand baggage can I carry on IndiGo?
4,Q05,IndiGo,cancellation_refund,How close to departure can I cancel an IndiGo ...


In [26]:
def category_hit(docs, expected_category):
    categories = [d.metadata.get('category') for d in docs]
    return int(expected_category in categories)


def airline_hit(docs, airline):
    airlines = [d.metadata.get('airline') for d in docs]
    return int(len(airlines) > 0 and all(a == airline for a in airlines))


def evaluate_retriever(method='hybrid', max_questions=None):
    rows = []
    subset = eval_df if max_questions is None else eval_df.head(max_questions)

    for _, row in subset.iterrows():
        airline = row['airline']
        question = row['question']
        expected_category = row['expected_category']

        try:
            if method == 'similarity':
                retriever = get_similarity_retriever(airline)
            elif method == 'mmr':
                retriever = get_mmr_retriever(airline)
            else:
                retriever = get_hybrid_retriever(airline)

            docs = retriever.invoke(question)
            rows.append({
                'question_id': row['question_id'],
                'airline': airline,
                'method': method,
                'airline_hit': airline_hit(docs, airline),
                'category_hit': category_hit(docs, expected_category),
                'retrieved_titles': ' | '.join([d.metadata.get('title','') for d in docs])
            })
        except Exception as e:
            rows.append({
                'question_id': row['question_id'],
                'airline': airline,
                'method': method,
                'airline_hit': 0,
                'category_hit': 0,
                'retrieved_titles': f'ERROR: {type(e).__name__}'
            })

    return pd.DataFrame(rows)

In [27]:
# Full evaluation can take time. Run this after indexing is complete.
similarity_results = evaluate_retriever('similarity')
mmr_results = evaluate_retriever('mmr')
hybrid_results = evaluate_retriever('hybrid')
#
all_results = pd.concat([similarity_results, mmr_results, hybrid_results], ignore_index=True)
summary = all_results.groupby('method')[['airline_hit','category_hit']].mean().round(3)
display(summary)
all_results.to_csv(PROJECT_ROOT / 'evaluation' / 'retrieval_results.csv', index=False)

,airline_hit,category_hit
method,,
hybrid,0.921,0.789
mmr,0.921,0.789
similarity,0.921,0.789


## 14. Controlled Failure Test

In [28]:
# This checks whether the model stays within the retrieved airline-policy context.
# The expected behavior is to say sufficient information was not found.
unsupported = graph.invoke({
    "airline": "IndiGo",
     "question": "Who won yesterday's cricket match?"
 })
print(unsupported['answer'].model_dump())

{'issue_type': 'non_policy_query', 'policy_summary': 'The retrieved policy documents pertain to IndiGo flight cancellation, refund, and conditions of carriage. They do not contain any information about cricket matches or sports results.', 'recommended_steps': ['For information about cricket match results, please refer to sports news websites, official cricket board announcements, or a search engine.'], 'source_documents': ['Flight Cancellation', 'Conditions of Carriage'], 'information_gap': True}


## 15. Demo Helper

In [29]:
def ask_flightwise(airline, question):
    result = graph.invoke({"airline": airline, "question": question})
    answer = result['answer']
    evidence = result['context']

    display(Markdown(f"### FlightWise Answer - {airline}"))
    display(Markdown(f"**Issue:** {answer.issue_type}"))
    display(Markdown(answer.policy_summary))

    display(Markdown("**Recommended steps**"))
    for step in answer.recommended_steps:
        display(Markdown(f"- {step}"))

    display(Markdown("**Sources used**"))
    for source in answer.source_documents:
        display(Markdown(f"- {source}"))

    if answer.information_gap:
        display(Markdown("**Information gap:** The retrieved context was not sufficient for a complete answer."))

    return answer, evidence

# Demo examples:
ask_flightwise("IndiGo", "My suitcase did not arrive. What should I do?")
# 
# ask_flightwise("Emirates", "How is checked baggage allowance calculated?")
# ask_flightwise("DGCA / MoCA", "What facilities should airlines provide after flight cancellation or delay?")

### FlightWise Answer - IndiGo

**Issue:** lost_baggage

IndiGo states that once checked‑in baggage is placed on the conveyor belt, the airline is not responsible for loss or theft. If baggage is not delivered, passengers must register a complaint at the arrival hall. For group bookings, all passengers in the booking must be present to file the complaint. Baggage will be delivered to the bearer of the baggage tag once any unpaid sums are settled.

**Recommended steps**

- Proceed to the airline’s baggage claim desk or lost‑baggage counter at the arrival airport as soon as you discover the suitcase is missing.

...

- If the baggage is declared lost, inquire about any applicable fees that need to be settled before the airline can deliver the bag to you, as IndiGo will release the bag only after any unpaid sums are cleared.

- Keep copies of all documents (baggage tag, boarding pass, complaint receipt) for future reference.

**Sources used**

- Conditions of Carriage

(PassengerSupportResponse(issue_type='lost_baggage', policy_summary='IndiGo states that once checked‑in baggage is placed on the conveyor belt, the airline is not responsible for loss or theft. If baggage is not delivered, passengers must register a complaint at the arrival hall. For group bookings, all passengers in the booking must be present to file the complaint. Baggage will be delivered to the bearer of the baggage tag once any unpaid sums are settled.', recommended_steps=['Proceed to the airline’s baggage claim desk or lost‑baggage counter at the arrival airport as soon as you discover the suitcase is missing.', 'Provide your baggage tag, boarding pass, and any identification details to the staff.', 'If you are traveling with others under the same booking, ensure all passengers are present at the arrival hall when filing the complaint, as IndiGo requires all members of the booking to be present.', 'Ask for a written complaint reference number and details on the next steps for lo

In [30]:
ask_flightwise("Air India", "If Air India cancels my flight, how can I request a refund?")

### FlightWise Answer - Air India

**Issue:** refund_request_flight_cancellation

If Air India cancels your flight, you can request a refund through the airline’s website or mobile app by using the “manage booking” section. Alternatively, you may choose to re‑accommodate (reschedule) via the self‑service re‑accommodation option on the website or by clicking the reschedule link sent to your registered email.

**Recommended steps**

- Log in to the Air India website or open the Air India mobile app.

...

- If you prefer to travel on a later date, use the “Self‑service Re‑accommodation” feature on the website or click the reschedule link in the cancellation email.

- After submitting, you will receive a receipt and case ID via email; you can track the status through the customer support portal.

**Sources used**

- Payment, Refund and Cancellation FAQ

(PassengerSupportResponse(issue_type='refund_request_flight_cancellation', policy_summary='If Air India cancels your flight, you can request a refund through the airline’s website or mobile app by using the “manage booking” section. Alternatively, you may choose to re‑accommodate (reschedule) via the self‑service re‑accommodation option on the website or by clicking the reschedule link sent to your registered email.', recommended_steps=['Log in to the Air India website or open the Air India mobile app.', 'Go to the “Manage Booking” (or “My Trips”) section.', 'Select the cancelled flight and choose the “Refund” option to submit a refund request.', 'If you prefer to travel on a later date, use the “Self‑service Re‑accommodation” feature on the website or click the reschedule link in the cancellation email.', 'After submitting, you will receive a receipt and case ID via email; you can track the status through the customer support portal.'], source_documents=['Payment, Refund and Cancellat

## 16. Limitations

1. Airline policy pages can change after indexing.
2. Some websites may block automated loading; failed URLs should be re-checked before the demo.
3. Policy wording may depend on route, fare family, booking channel and jurisdiction.
4. The prototype is for academic demonstration and is not legal advice.
5. Retrieval quality depends on the source corpus, chunking and embedding/retrieval choices.

## 17. AI Usage Disclosure

AI tools were used as a helper for brainstorming the project framing, reviewing documentation, checking code organization, and debugging suggestions. The implementation is based primarily on the syntax and patterns taught in the six in-class notebooks provided by the instructor. The group reviewed the final notebook, dataset choices, prompts, evaluation questions and presentation content and is responsible for understanding and explaining the submitted work.

## 18. Gradio Demo

A simple web UI on top of the LangGraph workflow from Section 11. Pick an airline (or DGCA / MoCA), type a passenger question, and FlightWise answers from the retrieved policy sources only. Run all earlier cells first (dataset, Chroma index, graph); the demo opens inline below and at a local URL.

In [31]:
import gradio as gr

airline_choices = sorted({d.metadata.get("airline") for d in all_docs if d.metadata.get("airline")})

example_questions = [
    ["IndiGo", "My suitcase did not arrive. What should I do?"],
    ["Emirates", "How is checked baggage allowance calculated?"],
    ["Air India", "If Air India cancels my flight, how can I request a refund?"],
    ["DGCA / MoCA", "What facilities should airlines provide after flight cancellation or delay?"],
]


def flightwise_gradio(airline, question):
    if not question or not question.strip():
        return "Please enter a question."

    result = graph.invoke({"airline": airline, "question": question})
    answer = result["answer"]

    output = f"### {answer.issue_type}\n\n{answer.policy_summary}\n\n"

    if answer.recommended_steps:
        output += "**Recommended steps**\n" + "\n".join(f"- {s}" for s in answer.recommended_steps) + "\n\n"

    if answer.source_documents:
        output += "**Sources used**\n" + "\n".join(f"- {s}" for s in answer.source_documents) + "\n\n"

    if answer.information_gap:
        output += "**Information gap:** The retrieved context was not sufficient for a complete answer.\n"

    return output


demo = gr.Interface(
    fn=flightwise_gradio,
    inputs=[
        gr.Dropdown(choices=airline_choices, value=airline_choices[0], label="Airline / Regulator"),
        gr.Textbox(lines=2, label="Passenger question", placeholder="e.g. My flight was cancelled. Can I get a refund?"),
    ],
    outputs=gr.Markdown(label="FlightWise answer"),
    title="FlightWise - Airline Passenger Policy Assistant",
    description="Ask a policy question. Answers are grounded only in the retrieved official airline / DGCA sources. Not legal advice.",
    examples=example_questions,
    flagging_mode="never",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
